In [3]:
df = spark.read.option("multiline", "true").json("Files/bronze/leads.json/leads_29-06-2026.json")
# df now is a Spark DataFrame containing JSON data from "Files/bronze/leads.json/leads_29-06-2026.json".
display(df)

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b04c4a9-86f2-4b94-b60d-1970aed57a38)

In [4]:
import html
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
# read every dated bronze file: leads_2026-06-29.json, etc.
BRONZE_PATH = "abfss://zohodata@onelake.dfs.fabric.microsoft.com/zoho.Lakehouse/Files/bronze/leads.json"
 
raw = (spark.read
       .option("multiline", "true")   # each file is ONE json object
       .json(BRONZE_PATH))
 
# the API wraps records in a "data" array -> explode to one row per lead
exploded = raw.select(F.explode("data").alias("r")).select("r.*")
 
print("raw lead rows read:", exploded.count())
exploded.printSchema()

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 6, Finished, Available, Finished, False)

raw lead rows read: 58
root
 |-- Company: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Lead_Status: string (nullable = true)
 |-- Modified_Time: string (nullable = true)
 |-- id: string (nullable = true)



In [5]:
@F.udf("string")
def html_decode(s):
    if s is None:
        return None
    out = s
    for _ in range(4):
        new = html.unescape(out)
        if new == out:
            break
        out = new
    return out.strip() if out else out

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 7, Finished, Available, Finished, False)

In [6]:
silver = (
    exploded
    # --- rename to the target schema (LeadID, Name, Email, Company, Status, ModifiedTime) ---
    .withColumn("LeadID",     F.col("id").cast("string"))
    .withColumn("FirstName",  html_decode(F.col("First_Name")))
    .withColumn("LastName",   html_decode(F.col("Last_Name")))
    .withColumn("Email",      F.lower(F.trim(F.col("Email"))))
    .withColumn("Company",    F.trim(F.col("Company")))
 
    # --- FullName: join first + last, drop the empties cleanly ---
    .withColumn("FullName",
        F.trim(F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))))
 
    # --- Status: kill the junk "true"/"false", keep real statuses ---
    .withColumn("Status",
        F.when(F.lower(F.trim(F.col("Lead_Status"))).isin("true", "false"), None)
         .otherwise(F.trim(F.col("Lead_Status"))))
 
    # --- parse the ISO8601-with-offset string into a real timestamp ---
    .withColumn("ModifiedTime",
        F.to_timestamp(F.col("Modified_Time")))
 
    # --- quality flag: mark obvious test/sample records (don't delete them) ---
    .withColumn("IsSample",
        F.when(
            F.col("LastName").rlike(r"(?i)\(sample\)") |
            F.col("Email").rlike(r"(?i)(noemail\.invalid|yopmail\.com)") |
            F.lower(F.col("FirstName")).isin("string", "test", "fgfhdgh") |
            F.lower(F.col("LastName")).isin("string", "test"),
            True
        ).otherwise(False))
 
    # --- audit column ---
    .withColumn("IngestedAt", F.current_timestamp())
 
    # --- final column set ---
    .select(
        "LeadID", "FirstName", "LastName", "FullName",
        "Email", "Company", "Status", "ModifiedTime",
        "IsSample", "IngestedAt"
    )
)

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 8, Finished, Available, Finished, False)

In [7]:
w = Window.partitionBy("LeadID").orderBy(F.col("ModifiedTime").desc_nulls_last())
silver = (silver
          .withColumn("rn", F.row_number().over(w))
          .filter("rn = 1")
          .drop("rn"))
 
print("clean unique leads:", silver.count())
silver.show(5, truncate=False)
 


StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 9, Finished, Available, Finished, False)

clean unique leads: 29
+-------------------+---------+-----------------+------------------------+-------------------------------+-------------------------+--------------------+-------------------+--------+--------------------------+
|LeadID             |FirstName|LastName         |FullName                |Email                          |Company                  |Status              |ModifiedTime       |IsSample|IngestedAt                |
+-------------------+---------+-----------------+------------------------+-------------------------------+-------------------------+--------------------+-------------------+--------+--------------------------+
|1332830000000457131|Chau     |Kitzman (Sample) |Chau Kitzman (Sample)   |chau-kitzman@noemail.invalid   |Creative Business Systems|Attempted to Contact|2026-06-15 07:17:15|true    |2026-06-29 12:41:02.813227|
|1332830000000457132|Theola   |Frey (Sample)    |Theola Frey (Sample)    |theola-frey@noemail.invalid    |Dal Tile Corporation     |Conta

In [8]:
TARGET = "silver_leads"
 
if spark.catalog.tableExists(TARGET):
    tgt = DeltaTable.forName(spark, TARGET)
    (tgt.alias("t")
        .merge(silver.alias("s"), "t.LeadID = s.LeadID")
        # only overwrite when the incoming record is actually newer
        .whenMatchedUpdateAll(condition="s.ModifiedTime > t.ModifiedTime")
        .whenNotMatchedInsertAll()
        .execute())
    print("MERGE complete into", TARGET)
else:
    (silver.write
        .format("delta")
        .saveAsTable(TARGET))
    print("created table", TARGET)
 
# quick verification
spark.sql(f"SELECT COUNT(*) AS rows, "
          f"SUM(CAST(IsSample AS INT)) AS samples FROM {TARGET}").show()

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 10, Finished, Available, Finished, False)

MERGE complete into silver_leads
+----+-------+
|rows|samples|
+----+-------+
|  29|     18|
+----+-------+



In [9]:
# =============================================================
# SILVER: Zoho CRM Leads -> silverzoho lakehouse (Tables/silver_leads)
# Add BOTH the Bronze lakehouse AND silverzoho to this notebook first.
# =============================================================

import html
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ---------- config: SET THESE ----------
WORKSPACE = "zohodata"          # your workspace name
BRONZE_LH = "zoho"              # <-- the LAKEHOUSE holding Files/bronze/ (NOT the bronzoho notebook)
SILVER_LH = "silverzoho"

BRONZE_PATH       = f"abfss://zohodata@onelake.dfs.fabric.microsoft.com/zoho.Lakehouse/Files/bronze/leads.json"
SILVER_TABLE_PATH = f"abfss://zohodata@onelake.dfs.fabric.microsoft.com/silverzoho.Lakehouse/Tables"

# ---------- read Bronze + explode ----------
raw = spark.read.option("multiline", "true").json(BRONZE_PATH)
exploded = raw.select(F.explode("data").alias("r")).select("r.*")
print("raw lead rows read:", exploded.count())

# ---------- fix triple-encoded HTML in names ----------
@F.udf("string")
def html_decode(s):
    if s is None:
        return None
    out = s
    for _ in range(4):
        new = html.unescape(out)
        if new == out:
            break
        out = new
    return out.strip() if out else out

# ---------- clean / reshape ----------
silver = (exploded
    .withColumn("LeadID",     F.col("id").cast("string"))
    .withColumn("FirstName",  html_decode(F.col("First_Name")))
    .withColumn("LastName",   html_decode(F.col("Last_Name")))
    .withColumn("Email",      F.lower(F.trim(F.col("Email"))))
    .withColumn("Company",    F.trim(F.col("Company")))
    .withColumn("FullName",   F.trim(F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))))
    .withColumn("Status",
        F.when(F.lower(F.trim(F.col("Lead_Status"))).isin("true", "false"), None)
         .otherwise(F.trim(F.col("Lead_Status"))))
    .withColumn("ModifiedTime", F.to_timestamp(F.col("Modified_Time")))
    .withColumn("IsSample",
        F.when(
            F.col("Last_Name").rlike(r"(?i)\(sample\)") |
            F.col("Email").rlike(r"(?i)(noemail\.invalid|yopmail\.com)") |
            F.lower(F.col("First_Name")).isin("string", "test", "fgfhdgh") |
            F.lower(F.col("Last_Name")).isin("string", "test"),
            True).otherwise(False))
    .withColumn("IngestedAt", F.current_timestamp())
    .select("LeadID", "FirstName", "LastName", "FullName", "Email",
            "Company", "Status", "ModifiedTime", "IsSample", "IngestedAt"))

# ---------- dedup: newest per LeadID ----------
w = Window.partitionBy("LeadID").orderBy(F.col("ModifiedTime").desc_nulls_last())
silver = silver.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
print("clean unique leads:", silver.count())

# ---------- MERGE into silverzoho/Tables/silver_leads ----------
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
    tgt = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
    (tgt.alias("t")
        .merge(silver.alias("s"), "t.LeadID = s.LeadID")
        .whenMatchedUpdateAll(condition="s.ModifiedTime > t.ModifiedTime")
        .whenNotMatchedInsertAll()
        .execute())
    print("MERGE complete into silverzoho.")
else:
    silver.write.format("delta").save(SILVER_TABLE_PATH)
    print("Created silver_leads in silverzoho.")

# ---------- verify ----------
result = spark.read.format("delta").load(SILVER_TABLE_PATH)
print("total rows in silver_leads:", result.count())
result.show(5, truncate=False)

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 11, Finished, Available, Finished, False)

raw lead rows read: 58
clean unique leads: 29
MERGE complete into silverzoho.
total rows in silver_leads: 29
+-------------------+---------+-----------------+------------------------+-------------------------------+-------------------------+--------------------+-------------------+--------+--------------------------+
|LeadID             |FirstName|LastName         |FullName                |Email                          |Company                  |Status              |ModifiedTime       |IsSample|IngestedAt                |
+-------------------+---------+-----------------+------------------------+-------------------------------+-------------------------+--------------------+-------------------+--------+--------------------------+
|1332830000000457131|Chau     |Kitzman (Sample) |Chau Kitzman (Sample)   |chau-kitzman@noemail.invalid   |Creative Business Systems|Attempted to Contact|2026-06-15 07:17:15|true    |2026-06-29 12:20:24.228218|
|1332830000000457132|Theola   |Frey (Sample)    |Th

In [10]:
# fill blanks so dashboards show "Unknown" instead of empty
silver = silver.fillna({"Company": "Unknown", "Status": "Unknown"})

# keep a row if it has an ID AND (a real name OR an email)
silver = silver.filter(
    F.col("LeadID").isNotNull() &
    (
        ((F.col("FullName").isNotNull()) & (F.col("FullName") != "")) |
        (F.col("Email").isNotNull())
    )
)

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 12, Finished, Available, Finished, False)

In [11]:
print("rows after filter:", silver.count())

StatementMeta(, 20ce9a2f-628a-4899-8fa8-23660e37b77e, 13, Finished, Available, Finished, False)

rows after filter: 29
